In [7]:
from co_occurrence import CoOccurrenceModel
from gensim.models import Word2Vec

import gzip
import json
import numpy as np
import pandas as pd
import re
import time

# Experiment 1: Word-context representation

In [8]:
# Khởi tạo corpus kích thước nhỏ
raw_corpus = [
    "Data mining discovers patterns in large datasets, uncovering associations and trends that may not be visible with traditional analysis techniques.",
    "Mining data helps find patterns in very large data sets, supporting decision-making processes in business, healthcare, and scientific research.",
    "Locality Sensitive Hashing finds similar documents efficiently by reducing high-dimensional vectors into compact signatures, enabling fast approximate nearest neighbor searches.",
    "MinHash approximates Jaccard similarity of sets, making it possible to compare massive collections of documents without constructing full pairwise similarity matrices.",
    "Efficient algorithms are crucial when working with massive datasets, as they allow practitioners to handle terabytes of text and multimedia data within reasonable time and memory constraints.",
    "Approximate methods trade accuracy for speed and scalability, which is essential when exact solutions are computationally infeasible in domains such as web search or large-scale recommendation.",
    "Hashing techniques are widely used in information retrieval systems, supporting fast indexing, deduplication, and similarity search across billions of documents."
]

# Hàm đếm số phần tử khác không trong word-context matrix
def count_non_zero_values(matrix: np.ndarray):
    rows, columns = matrix.shape
    count = 0
    for i in range(rows):
        for j in range(columns):
            if matrix[i, j] != 0.0:
                count += 1
    return count

# Hàm đo sparsity trên word-context matrix
def sparsity(matrix: np.ndarray):
    r, c = matrix.shape
    return round((1 - (count_non_zero_values(matrix) / (r * c))) * 100, 2)

# Train model
context_window = [1, 2, 5]

models = {
    1: CoOccurrenceModel(),
    2: CoOccurrenceModel(),
    5: CoOccurrenceModel()
}

vocab_size_list = []
matrix_shape_list = []
non_zeros_list = []
sparsity_list = []

for k in context_window:
    models[k].fit(raw_corpus, k)
    vocab_size_list.append(models[k].vocab_size)
    matrix_shape_list.append(models[k].matrix.shape)
    non_zeros_list.append(count_non_zero_values(models[k].matrix))
    sparsity_list.append(sparsity(models[k].matrix))

# Lập bảng so sánh window theo kích thước vocab, matrix, số phần tử khác 0 trong matrix và độ thưa
df = pd.DataFrame(
    [vocab_size_list, matrix_shape_list, non_zeros_list, sparsity_list],
    index = ['Vocabulary size', 'Matrix size', 'Number of non-zero values', 'Sparsity (%)'],
    columns = context_window
)
df.rename_axis('k', inplace=True)
print(df)

                                    1           2           5
k                                                            
Vocabulary size                   116         116         116
Matrix size                (116, 116)  (116, 116)  (116, 116)
Number of non-zero values         286         562        1256
Sparsity (%)                    97.87       95.82       90.67


Khi tăng context window, số phần tử khác 0 của word-context matrix ngày càng tăng, trong khi độ thưa (sparsity) có xu hướng giảm, khi k = 1 thì sparsity bằng 97.87% nhưng khi k = 5 thì sparsity chỉ còn 90.67%.

In [10]:
# Lấy 5 từ có similarity cao nhất với từ 'data'
for k in context_window:
    print(models[k].most_similar('data', 5))

[('discovers', np.float64(0.4714045207910316)), ('find', np.float64(0.2357022603955158)), ('making', np.float64(0.2357022603955158)), ('reasonable', np.float64(0.2357022603955158)), ('very', np.float64(0.2357022603955158))]
[('patterns', np.float64(0.5715476066494082)), ('helps', np.float64(0.3872983346207417)), ('time', np.float64(0.3872983346207417)), ('in', np.float64(0.26352313834736496)), ('decision-making', np.float64(0.2581988897471611))]
[('helps', np.float64(0.7381017469576844)), ('large', np.float64(0.7149000896634314)), ('mining', np.float64(0.6315089779506726)), ('find', np.float64(0.5763708787148328)), ('very', np.float64(0.5637345210021217))]


Khi tăng giá trị context window, similarity giữa context word vector và target word vector ngày càng tăng. Điều này cho thấy khi mô hình học được lượng ngữ cảnh nhiều hơn thì lượng thông tin tiếp nhận sẽ tăng và có thể tìm ra các từ có thể similarity cao hơn.

# Từ co-occurence matrix đến embedding #
**Nếu vocabulary có 100,000 từ thì kích thước word-context matrix sẽ rất lớn và thưa, cỡ 100,000 x 100,000, gây ra sự tiêu tốn trong lưu trữ bộ nhớ và việc tính toán sẽ chậm hơn nhiều, nhất là semantic similarity.**

# Khác biệt giữa Skip-gram và CBOW
**Với CBOW, input là các từ xét trong cùng 1 context cụ thể, trong khi target của nó là 1 từ mục tiêu dự đoán trên context đang xét.**


**Với Skip-gram thì ngược lại, input là một target word, trong khi target của nó là dự đoán các context words dựa theo thông tin của target word đã cho.**

# Experiment 2: Word2Vec

In [17]:
data_path = '../data/c4-train.00000-of-01024-30K.json.gz'

# Đọc dữ liệu
raw_corpus = []

with gzip.open(data_path, 'rt', encoding = 'utf-8') as f:
    for line in f:
        data = json.loads(line)
        raw_corpus.append(data['text'])

print(f'Kích thước corpus: {len(raw_corpus)}')

# Tiền xử lý dữ liệu
## Hàm tách câu
def split_sentences(document):
    return [
        sentence.strip()
        for sentence in re.split(r'(?<=[.!?])\s+|\n+', document)
        if sentence.strip()
    ]

## Hàm tokenize
def tokenize(sentence):
    sentence = sentence.lower()
    return re.findall(
        r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
        sentence
    )

# Tập document đã được tokenize
tokenized_documents = []
# Tập các câu đã được tokenize
tokenized_sentences = []
# Tập từ vựng
vocabulary = set()
# Bước tokenize
for document in raw_corpus:
    document_tokens = []

    for sentence in split_sentences(document):
        tokens = tokenize(sentence)

        if tokens:
            tokenized_sentences.append(tokens)
            vocabulary.update(tokens)
            document_tokens.extend(tokens)

    if document_tokens:
        tokenized_documents.append(document_tokens)

vocabulary = sorted(vocabulary)

print(f'Số câu: {len(tokenized_sentences)}')
print(f'Kích thước vocabulary: {len(vocabulary)}')

Kích thước corpus: 30000
Số câu: 634941
Kích thước vocabulary: 227644


In [18]:
model_w2v = Word2Vec(
    sentences = tokenized_sentences,
    vector_size = 100,
    window = 5,
    min_count = 2,
    epochs = 10,
    sg = 0,
    seed = 42
)

In [19]:
inspect_word = ['doctor', 'hospital', 'patient', 'disease', 'computer', 'football', 'banana']
top_5_list = []
for word in inspect_word:
    top_5 = model_w2v.wv.most_similar(word, topn = 5)
    top_5_word = [w[0] for w in top_5]
    top_5_list.append(top_5_word)

df_top_5 = pd.DataFrame(
    top_5_list,
    index = inspect_word,
    columns = [f'top_{k}' for k in range(1, 6)]
)

df_top_5

,top_1,top_2,top_3,top_4,top_5
doctor,dentist,physician,veterinarian,therapist,pharmacist
hospital,clinic,hospitals,clinics,daycare,rehab
patient,diagnosis,patients,physician,therapist,medication
disease,diseases,infection,chronic,lung,cancer
computer,desktop,device,laptop,browser,pc
football,baseball,soccer,basketball,hockey,rugby
banana,caramelized,pineapple,sausage,tomato,mozzarella


# Experiment 3: Context window

In [20]:
window_size = [2, 5, 10]
word_pair = [
    ('doctor', 'physician'),
    ('doctor', 'hospital'),
    ('cat', 'dog')
]
window_results = []

for k in window_size:
    model = Word2Vec(
        sentences = tokenized_sentences,
        vector_size = 100,
        min_count = 2,
        window = k,
        epochs = 10,
        seed = 42
    )
    window_similarity = []

    for first_word, second_word in word_pair:
        window_similarity.append(model.wv.similarity(first_word, second_word))

    window_results.append(window_similarity)

df_window = pd.DataFrame(
    list(zip(*window_results)),
    columns = [f'Window {k}' for k in window_size],
    index = ['doctor-physician', 'doctor-hospital', 'cat-dog']
)

df_window

,Window 2,Window 5,Window 10
doctor-physician,0.794375,0.785188,0.781345
doctor-hospital,0.480453,0.452717,0.488529
cat-dog,0.742665,0.780733,0.796033


# Experiment 4: Embedding dimension

In [22]:
dimensions = [50, 100]

training_time = []
model_size = []
dim_similarity = []
analogy = []
models_by_dimension = {}

for dim_value in dimensions:
    start = time.perf_counter()

    model = Word2Vec(
        sentences = tokenized_sentences,
        vector_size = dim_value,
        window = 5,
        min_count = 2,
        sg = 1,
        workers = 1,
        seed = 42,
        epochs = 10
    )
    elapsed = time.perf_counter() - start
    training_time.append(elapsed)
    models_by_dimension[dim_value] = model

    model_size.append(model.wv.vectors.nbytes)

    dim_similarity_pair = []
    for first_word, second_word in word_pair:
        dim_similarity_pair.append(model.wv.similarity(
            first_word, second_word
        ))
    dim_similarity.append(dim_similarity_pair)

    analogy.append(model.wv.most_similar(
        positive = ['king', 'woman'],
        negative = ['man'],
        topn = 5
    ))

df_eval_dim = pd.DataFrame(
    [training_time, model_size],
    index = ['Training time', 'Model size'],
    columns = dimensions
)
df_eval_dim.rename_axis('Embedding dimension', inplace=True)

df_eval_dim

,50,100
Embedding dimension,,
Training time,7.490138e+02,7.633053e+02
Model size,2.178500e+07,4.357000e+07


In [23]:
df_similarity_dim = pd.DataFrame(
    list(zip(*dim_similarity)),
    columns = dimensions,
    index = ['doctor-physician', 'doctor-hospital', 'cat-dog']
)
df_similarity_dim.rename_axis('Embedding dimension', inplace=True)
df_similarity_dim

,50,100
Embedding dimension,,
doctor-physician,0.825215,0.731052
doctor-hospital,0.623291,0.563873
cat-dog,0.860076,0.780533


In [24]:
analogy_rows = []

for dimension, results in zip(dimensions, analogy):
    for rank, (word, similarity) in enumerate(results, start=1):
        analogy_rows.append({
            'Dimension': dimension,
            'Rank': rank,
            'Top-5 word': word,
            'Similarity': similarity
        })

df_analogy = pd.DataFrame(analogy_rows)
df_analogy

,Dimension,Rank,Top-5 word,Similarity
0,50,1,queen,0.888257
1,50,2,victoria,0.714008
2,50,3,prince,0.712427
3,50,4,albert,0.706699
4,50,5,royal,0.705625
5,100,1,queen,0.780345
6,100,2,queen's,0.560841
7,100,3,duke,0.549678
8,100,4,duchess,0.537112
9,100,5,lila,0.532188


In [25]:
medical_words = ['doctor', 'hospital', 'patient', 'disease']
sports_words = ['football', 'baseball', 'basketball', 'hockey']
test_words = {
    'physician': 'medical',
    'clinic': 'medical',
    'infection': 'medical',
    'soccer': 'sports',
    'nba': 'sports'
}

def cosine_similarity(vector1, vector2):
    denominator = np.linalg.norm(vector1) * np.linalg.norm(vector2)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector1, vector2) / denominator)

downstream_rows = []
for dimension, model in models_by_dimension.items():
    medical = [word for word in medical_words if word in model.wv]
    sports = [word for word in sports_words if word in model.wv]
    medical_centroid = np.mean([model.wv[word] for word in medical], axis=0)
    sports_centroid = np.mean([model.wv[word] for word in sports], axis=0)

    for word, expected in test_words.items():
        if word not in model.wv:
            downstream_rows.append({
                'dimension': dimension, 'word': word,
                'expected': expected, 'predicted': 'OOV', 'correct': False
            })
            continue
        medical_score = cosine_similarity(model.wv[word], medical_centroid)
        sports_score = cosine_similarity(model.wv[word], sports_centroid)
        predicted = 'medical' if medical_score >= sports_score else 'sports'
        downstream_rows.append({
            'dimension': dimension, 'word': word,
            'expected': expected, 'predicted': predicted,
            'medical_score': medical_score, 'sports_score': sports_score,
            'correct': predicted == expected
        })

df_downstream_detail = pd.DataFrame(downstream_rows)
df_downstream_detail

,dimension,word,expected,predicted,medical_score,sports_score,correct
0,50,physician,medical,medical,0.827107,0.309382,True
1,50,clinic,medical,medical,0.810226,0.384891,True
2,50,infection,medical,medical,0.770317,0.195619,True
3,50,soccer,sports,sports,0.276256,0.886889,True
4,50,nba,sports,sports,0.238656,0.798841,True
5,100,physician,medical,medical,0.741741,0.258897,True
6,100,clinic,medical,medical,0.759265,0.318941,True
7,100,infection,medical,medical,0.659733,0.168261,True
8,100,soccer,sports,sports,0.255145,0.808296,True
9,100,nba,sports,sports,0.257922,0.723724,True


In [26]:
df_downstream_summary = (
    df_downstream_detail.assign(
        evaluated=lambda data: data['predicted'] != 'OOV'
    )
    .groupby('dimension')
    .agg(evaluated=('evaluated', 'sum'), correct=('correct', 'sum'))
    .assign(accuracy=lambda data: data['correct'] / data['evaluated'])
)
df_downstream_summary

,evaluated,correct,accuracy
dimension,,,
50,5,5,1.0
100,5,5,1.0
